# Magnitude Homogenization

[Magnitude Homogenization](https://gemsciencetools.github.io/quakeT/contents/glossary.html#term-Magnitude-homogenization) is one of the fundamental steps in seismic hazard analysis. The process involves combining earthquake data from multiple catalogues into a single consistent [earthquake catalogue](https://gemsciencetools.github.io/quakeT/contents/glossary.html#term-Earthquake-catalogue) by carefully evaluating available bulletins, identifying duplicate events across different sources, and applying empirical conversion relationships to transform various native magnitude scales into a common target magnitude scale [[9]](references.html#ref-weatherill-2016).

There is a [homogenisor module in oq-mbtk](https://github.com/GEMScienceTools/oq-mbtk/blob/master/openquake/cat/isc_homogenisor.py) that was developed for the database of the [International Seismological Center (ISC)](https://www.isc.ac.uk/), but can be used for any other compiled database to create a magnitude-homogeneous global [earthquake catalogue](https://gemsciencetools.github.io/quakeT/contents/glossary.html#term-Earthquake-catalogue). Please note that the module contains not only the reviewed and reprocessed time/location (origin) and magnitude solutions for each event, but also the original solutions from all of the contributing local recording networks [[9]](references.html#ref-weatherill-2016). This homogenisor supports several agency-magnitude scales (e.g., [surface-wave magnitude](https://gemsciencetools.github.io/quakeT/contents/glossary.html#term-Surface-wave-magnitude-M-s) ($M_s$), [body-wave magnitude](https://gemsciencetools.github.io/quakeT/contents/glossary.html#term-Body-wave-magnitude-m-b) ($m_b$), and [moment magnitude](https://gemsciencetools.github.io/quakeT/contents/glossary.html#term-Moment-magnitude-M-w) ($M_w$)) and converts them to a standardized $M_w$ using empirical relations based on their hierarchical criteria for selection of origin and magnitude for homogenizing a global earthquake catalogue.

In [1]:
import pandas as pd
import numpy as np

from openquake.cat import isc_homogenisor

Here, we define some case rules for the homogenisation. If you want to know more about `isc_homogenisor` and their rules, please refer to [homogenisor module in oq-mbtk](https://github.com/GEMScienceTools/oq-mbtk/blob/master/openquake/cat/isc_homogenisor.py).

Briefly, the block below initializes the logic for converting different magnitude scales reported by the [International Seismological Center (ISC)](https://www.isc.ac.uk/) into a standardized $M_w$ using the ISC-GEM methodology. 

**Ms rule**
- Agency: ISC
- Model: Converts $M_s$ to $M_w$ using the General Orthogonal Regression (GOR) model.
- Uncertainty: Applies a specific sigma model (`ISCGORMs_toGCMTMw_Sigma`) to track measurement errors.

**mb rule**
- Agency: ISC
- Model: Converts $m_b$ to $M_w$ using the General Orthogonal Regression (GOR) model tailored for body waves.
- Uncertainty: Applies the corresponding $m_b$ sigma model (`ISCGORmb_toGCMTMw_Sigma`) to ensure consistent error propagation.

In [2]:
homog_rules = {
    'Ms': isc_homogenisor.MagnitudeConversionRule(
        'ISC', 'Ms', 
        isc_homogenisor.ISCGORMs_toGCMTMw, 
        isc_homogenisor.ISCGORMs_toGCMTMw_Sigma
    ),
    'mb': isc_homogenisor.MagnitudeConversionRule(
        'ISC', 'mb', 
        isc_homogenisor.ISCGORmb_toGCMTMw, 
        isc_homogenisor.ISCGORmb_toGCMTMw_Sigma
    )
}

Below, a mock dataset representing raw earthquake data with different scales and original uncertainties (sigma) is created.

In [3]:
raw_data = {
    'event_id': [1, 2, 3, 4],
    'orig_scale': ['Ms', 'mb', 'Ms', 'mb'],
    'orig_mag': [7.2, 5.4, 6.1, 4.8],
    'orig_sigma': [0.1, 0.1, 0.1, 0.1]
}

df = pd.DataFrame(raw_data)
display(df)

,event_id,orig_scale,orig_mag,orig_sigma
0,1,Ms,7.2,0.1
1,2,mb,5.4,0.1
2,3,Ms,6.1,0.1
3,4,mb,4.8,0.1


You have a sample dataset and defined conversion rules. Now, you can run the code snippet below and homogenize your catalogue.

In [4]:
def homogenize(row):
    """
    Applies conversion rules to convert original magnitude to Mw.
    Handles list outputs from GOR models and propagates uncertainty.
    """
    rule = homog_rules.get(row['orig_scale'])
    
    if rule:
        new_mag, new_sigma = rule.convert_value(row['orig_mag'], row['orig_sigma'])
        
        if isinstance(new_mag, (list, np.ndarray)):
            new_mag = new_mag[0]
            
        return pd.Series([round(float(new_mag), 2), round(new_sigma, 3)])

df[['Mw_Unified', 'Sigma_Unified']] = df.apply(homogenize, axis=1)

print("\nHomogenized Catalogue:")
display(df)


Homogenized Catalogue:


,event_id,orig_scale,orig_mag,orig_sigma,Mw_Unified,Sigma_Unified
0,1,Ms,7.2,0.1,7.25,0.228
1,2,mb,5.4,0.1,5.66,0.330
2,3,Ms,6.1,0.1,6.22,0.211
3,4,mb,4.8,0.1,4.83,0.330
